# Stage 4 projector architecture and optimization sweep

This Colab asks whether the retained shallow `768 → 512 → 384` Stage 4
projector and conservative AdamW settings underfit the full
cached-SPECTER2-to-**raw frozen-AE latent** mapping.

Scientific controls are fixed: the dataset and ordered splits, published
Stage 3/4 text cache, frozen branch-matched Stage 1 autoencoders, raw
384-dimensional latent target, decoder, loss weights, and evaluation
conventions. The production projector is not edited. Larger projectors
live only in `neurovlm.experiments.stage4_projectors`.

The causal sequence is:

1. identical 32-example capacity gates for every architecture;
2. raw-loss architecture/LR/weight-decay/dropout sweep;
3. a separate scheduler comparison at selected optimizer settings;
4. an optional, explicitly secondary standardized-latent-loss sweep;
5. validation-only ranking and Pareto analysis;
6. test evaluation only for final validation-selected configurations.

`FAST_SWEEP` is a 25-epoch PubMed-mixed screen. `FULL_SWEEP` consumes
optimizer settings selected by a completed fast sweep and uses the full
branch and epoch budget.

## 1. Drive, repository, and immutable commit

Set `NEUROVLM_PINNED_COMMIT` to the 40-character commit containing this
notebook. Colab refuses a branch-only checkout. `REPO_REF` is recorded as
a human-readable development selector, while the immutable SHA is the
executed source of truth.

In [ ]:
from pathlib import Path
import os, re, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

REPO_URL = "https://github.com/neurovlm/neurovlm.git"
REPO_REF = "neurovlm_experiments"
REPO_DIR = Path("/content/neurovlm" if IN_COLAB else Path.cwd()).resolve()
PINNED_COMMIT = os.environ.get("NEUROVLM_PINNED_COMMIT", "").strip()
if not PINNED_COMMIT and not IN_COLAB and (REPO_DIR / ".git").is_dir():
    PINNED_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
    ).strip()
if not re.fullmatch(r"[0-9a-fA-F]{40}", PINNED_COMMIT):
    raise ValueError(
        "Set NEUROVLM_PINNED_COMMIT to the immutable 40-character commit "
        "that contains this notebook."
    )

if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if IN_COLAB:
    subprocess.run(["git", "fetch", "--all", "--tags", "--prune"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO_DIR, check=True)
RESOLVED_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
if RESOLVED_COMMIT != PINNED_COMMIT:
    raise RuntimeError(
        f"Pinned commit mismatch: requested {PINNED_COMMIT}, got {RESOLVED_COMMIT}"
    )
if IN_COLAB:
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError("Strict Colab provenance requires a clean checkout")
print("Repository:", REPO_DIR)
print("Branch selector:", REPO_REF)
print("Pinned commit:", RESOLVED_COMMIT)

## 2. Install the pinned repository

Lion is intentionally not installed: it is not an allowed dependency in
`pyproject.toml`, so every causal optimizer comparison remains AdamW.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{REPO_DIR}[metrics,viz,notebook]",
    ],
    check=True,
)
source_path = str(REPO_DIR / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)
os.environ["PYTHONPATH"] = source_path + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(REPO_DIR)
print("Installed pinned editable source:", source_path)

## 3. Experiment configuration

Exactly one mode must be active. Fast mode runs the required
`retained_mlp`, `deep_mlp`, and `residual_1024` grid on
`mixed_to_pubmed` first. Full mode requires the
`fast_selected_optimizer_settings.json` artifact from a completed fast
sweep (or an explicit `SELECTED_OPTIMIZER_SETTINGS` override).

The primary loss is always:

`MSE(predicted_raw_latent, target_raw_latent) + MSE(decoded_prediction, target_volume)`.

In [ ]:
FAST_SWEEP = True
FULL_SWEEP = False
assert FAST_SWEEP ^ FULL_SWEEP, "Select exactly one sweep mode"

ALL_BRANCHES = [
    "mixed_to_pubmed",
    "mixed_to_nilearn",
    "mixed_to_neurovault",
]
BRANCHES_TO_RUN = ["mixed_to_pubmed"] if FAST_SWEEP else ALL_BRANCHES
ALL_ARCHITECTURES = [
    "retained_mlp", "wider_mlp", "deep_mlp", "layernorm_deep",
    "residual_1024", "residual_2048", "gated_residual",
]
FAST_ARCHITECTURES = ["retained_mlp", "deep_mlp", "residual_1024"]
FAST_LEARNING_RATES = [5e-5, 1e-4, 3e-4, 1e-3]
FAST_WEIGHT_DECAYS = [0.0, 1e-5, 1e-4]
FAST_DROPOUTS = [0.0, 0.1]
SCHEDULERS = [
    "constant", "warmup_cosine", "reduce_on_plateau", "cosine_restarts"
]

SEED = 42
PROJECTOR_SEED = 42
FAST_EPOCHS = 25
FULL_EPOCHS = 100
EPOCHS = FAST_EPOCHS if FAST_SWEEP else FULL_EPOCHS
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
NUM_WORKERS = 8 if IN_COLAB else 0
PREFETCH_FACTOR = 4
GRADIENT_CLIP = 1.0
# Keep the optimizer-step budget identical within each causal phase.
# Validation best-checkpoint selection still occurs every epoch.
EARLY_STOPPING_PATIENCE = None
EARLY_STOPPING_MIN_DELTA = 0.0
AMP_DTYPE = "auto"
RESIDUAL_2048_BLOCKS = 2
VALIDATE_EVERY_EPOCHS = 1
SEMANTIC_MAX_EXAMPLES = 1024
SEMANTIC_NEIGHBORS = 10
FULL_DATA_LIMIT = None

TINY_OVERFIT_N = 32
TINY_OVERFIT_STEPS = 1500
TINY_EVAL_EVERY = 10
TINY_LEARNING_RATE = 1e-3
TINY_PASS_FRACTION = 0.95

RUN_SCHEDULER_COMPARISON = True
RUN_SECONDARY_STANDARDIZED_SWEEP = False
FAST_SELECTED_PER_ARCHITECTURE = 1
FULL_SELECTED_SETTING_COUNT = 2
FINALISTS_PER_BRANCH = 2
PARETO_EPSILON = 1e-6
PARETO_OBJECTIVES = {
    "val_top5_dice": "max",
    "val_spatial_corr": "max",
    "val_semantic_normalized_auc": "max",
    "val_latent_variance_ratio": "max",
    "val_global_explained_variance": "max",
}
RANK_COLUMNS = list(PARETO_OBJECTIVES)
COMPLEXITY_DICE_GAIN_THRESHOLD = 0.01

SELECTED_OPTIMIZER_SETTINGS = None
FAST_RESULTS_DIR_FOR_FULL = None
DRIVE_OUTPUT_BASE = Path(
    "/content/drive/MyDrive/neurovlm/stage4_projector_architecture_optimization_sweep"
    if IN_COLAB
    else REPO_DIR / "runs" / "stage4_projector_architecture_optimization_sweep"
)
RESUME_EXPERIMENT_DIR = None
AUTO_RESUME_ACTIVE = True

assert EPOCHS >= 20 if FAST_SWEEP else EPOCHS == FULL_EPOCHS
assert BRANCHES_TO_RUN[0] == "mixed_to_pubmed"
assert set(FAST_ARCHITECTURES) == {
    "retained_mlp", "deep_mlp", "residual_1024"
}

## 4. Environment, determinism, architecture table, and root artifacts

BF16 is used on compatible A100/H100 GPUs, FP16 is the CUDA fallback,
and float32 is the CPU fallback. Parameter counts and a transparent
projector-only activation-memory estimate are saved before training;
measured peak CUDA memory is logged per epoch and run.

In [ ]:
import copy, csv, hashlib, importlib.metadata, itertools, json, math
import platform, random, tempfile, time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from matplotlib import pyplot as plt
from torch import nn
from torch.utils.data import DataLoader

from neurovlm import retrieval_resources as rr
from neurovlm.atlas_free_dataset import AtlasFreeCNNDataProvider
from neurovlm.atlas_free_text import (
    AtlasFreeContrastiveCollator, AtlasFreeTextEmbeddingLookup,
)
from neurovlm.cnn import (
    CNNTextToBrainModel, GenerativeTextToAELatent, autoencoder_from_payload,
)
from neurovlm.evaluation.spatial import reconstruction_metrics
from neurovlm.evaluation.text_to_brain_audit import (
    ae_ceiling_bypass, audit_pairings, audit_raw_latent_path,
    audit_text_preprocessing, autoencoder_identity, frozen_ae_determinism,
)
from neurovlm.experiments.stage4_latent_ablation import (
    LatentTransform, encode_stage1_latents, latent_ablation_metrics,
    resolve_amp_dtype, split_fingerprint, text_cache_identity,
)
from neurovlm.experiments.stage4_projectors import (
    ActivationMonitor, PROJECTOR_NAMES, ProjectorBuildConfig,
    architecture_record, build_scheduler, build_stage4_projector,
    clone_trainable_parameters, count_parameters,
    detect_training_pathologies, gradient_diagnostics,
    parameter_update_norm, pareto_front, projector_checkpoint_metadata,
    projector_definition, step_scheduler,
    validate_projector_checkpoint_metadata,
)
from neurovlm.pipelines import (
    atomic_write_csv, atomic_write_json, environment_provenance,
    git_provenance, sha256_file, sha256_state_dict, sha256_value,
)
from neurovlm.semantic_evaluation import evaluate_semantic_neighbor_retrieval
from neurovlm.training.text_to_brain import (
    _autoencoder_state_provenance, _text_cache_provenance,
    _validate_recorded_autoencoder_state, _validate_recorded_text_cache,
)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)

seed_everything(SEED)
torch.set_float32_matmul_precision("high")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MIXED_PRECISION_DTYPE = resolve_amp_dtype(DEVICE, AMP_DTYPE)
packages = [
    "neurovlm", "torch", "numpy", "pandas", "matplotlib", "nilearn",
    "nibabel", "huggingface-hub", "transformers",
]
ENVIRONMENT = {
    **environment_provenance(packages),
    "python_full": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_capability": (
        torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
    ),
    "bf16_supported": (
        torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
    ),
    "training_amp_dtype": str(MIXED_PRECISION_DTYPE),
    "git": git_provenance(REPO_DIR),
    "configured_branch_selector": REPO_REF,
    "pinned_commit": PINNED_COMMIT,
    "resolved_commit": RESOLVED_COMMIT,
    "optimizer": "AdamW",
    "lion_status": "not compared; Lion is not an allowed project dependency",
}

def utc_stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def resolve_experiment_root():
    DRIVE_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    pointer = DRIVE_OUTPUT_BASE / "ACTIVE_EXPERIMENT.json"
    if RESUME_EXPERIMENT_DIR is not None:
        root = Path(RESUME_EXPERIMENT_DIR)
    elif AUTO_RESUME_ACTIVE and pointer.exists():
        active = json.loads(pointer.read_text())
        candidate = Path(active["path"])
        root = (
            candidate
            if active.get("state") != "completed" and candidate.exists()
            else DRIVE_OUTPUT_BASE / utc_stamp()
        )
    else:
        root = DRIVE_OUTPUT_BASE / utc_stamp()
    root.mkdir(parents=True, exist_ok=True)
    atomic_write_json(
        pointer,
        {"path": str(root), "state": "running", "updated_at": utc_stamp()},
    )
    return root, pointer

EXPERIMENT_ROOT, ACTIVE_POINTER = resolve_experiment_root()
ARCHITECTURE_RECORDS = []
for name in ALL_ARCHITECTURES:
    for dropout in sorted(set(FAST_DROPOUTS)):
        config = ProjectorBuildConfig(
            name=name, dropout=dropout,
            residual_2048_blocks=RESIDUAL_2048_BLOCKS,
        )
        projector = build_stage4_projector(
            name, dropout=dropout,
            residual_2048_blocks=RESIDUAL_2048_BLOCKS,
        )
        ARCHITECTURE_RECORDS.append(
            architecture_record(
                config, projector, batch_size=BATCH_SIZE,
                dtype=MIXED_PRECISION_DTYPE,
            )
        )
atomic_write_json(
    EXPERIMENT_ROOT / "architecture_definitions.json",
    ARCHITECTURE_RECORDS,
)
atomic_write_csv(
    EXPERIMENT_ROOT / "parameter_count_table.csv",
    [
        {
            "architecture": row["name"],
            "dropout": row["dropout"],
            "parameter_count": row["parameter_count"],
            **{
                f"activation_{key}": value
                for key, value in row["activation_memory"].items()
            },
        }
        for row in ARCHITECTURE_RECORDS
    ],
)
atomic_write_json(EXPERIMENT_ROOT / "environment.json", ENVIRONMENT)
print(json.dumps(ENVIRONMENT, indent=2, default=str))
print("Experiment root:", EXPERIMENT_ROOT)

## 5. Fixed branch resources and strict scientific provenance

Each branch uses the existing published frozen AE and matching Stage 3
semantic encoder. Ordered split identities, pairings, text cache, text
preprocessing, frozen AE tensor state, and raw-latent decoder path are
audited and bound into every checkpoint. Cached training latents are
accepted only when both ordered split and encoder hashes match.

In [ ]:
BRANCH_SPECS = {
    "mixed_to_pubmed": {"domain": "pubmed", "variant": "mixed_baseline", "stage1": "1A", "ae_variant": "mixed"},
    "mixed_to_nilearn": {"domain": "nilearn", "variant": "mixed_baseline", "stage1": "1A", "ae_variant": "mixed"},
    "mixed_to_neurovault": {"domain": "neurovault", "variant": "mixed_baseline", "stage1": "1A", "ae_variant": "mixed"},
}
for name, spec in BRANCH_SPECS.items():
    spec["branch"] = name
unknown = sorted(set(BRANCHES_TO_RUN) - set(BRANCH_SPECS))
if unknown:
    raise ValueError(f"Unknown branch selectors: {unknown}")

def freeze_module(module):
    module.eval()
    for parameter in module.parameters():
        parameter.requires_grad_(False)
    return module

def make_loader(dataset, lookup, *, batch_size, shuffle, seed):
    lookup.validate_dataset(dataset.rows)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        collate_fn=AtlasFreeContrastiveCollator(lookup, (36, 45, 38)),
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        generator=torch.Generator().manual_seed(seed),
        **({"prefetch_factor": PREFETCH_FACTOR} if NUM_WORKERS > 0 else {}),
    )

def load_branch_resources(branch_name):
    spec = BRANCH_SPECS[branch_name]
    filename = rr.CNN_AUTOENCODER_FILENAMES[spec["ae_variant"]]
    ae_path = Path(
        rr._download_from_hf(
            rr.ATLAS_FREE_CNN_MODEL_REPO_ID, filename, repo_type="model"
        )
    )
    payload = torch.load(ae_path, map_location="cpu", weights_only=True)
    autoencoder = freeze_module(autoencoder_from_payload(payload))
    autoencoder.encoder.eval()
    autoencoder.decoder.eval()
    provider = AtlasFreeCNNDataProvider(
        domain=spec["domain"], limit=FULL_DATA_LIMIT
    )
    semantic_model = freeze_module(rr._load_cnn_contrastive(branch_name))
    return spec, ae_path, autoencoder, provider, semantic_model

def build_branch_provenance(
    spec, ae_path, autoencoder, provider, lookup, audit_dir
):
    ae_source = _autoencoder_state_provenance(
        {
            "kind": "released",
            "path": str(ae_path.resolve()),
            "sha256": sha256_file(ae_path),
            "branch": spec["branch"],
            "domain": spec["domain"],
            "stage1": spec["stage1"],
            "variant": spec["variant"],
            "loader_variant": spec["ae_variant"],
        },
        autoencoder,
    )
    cache_source = _text_cache_provenance(lookup)
    _validate_recorded_autoencoder_state(ae_source, autoencoder)
    _validate_recorded_text_cache(
        cache_source, _text_cache_provenance(lookup)
    )
    text_audit = audit_text_preprocessing(lookup)
    if not text_audit["passed"]:
        raise RuntimeError(f"Text-cache convention mismatch: {text_audit}")
    pairings = {}
    for split in ("train", "val", "test"):
        dataset = getattr(provider, split)
        pairings[split] = audit_pairings(
            dataset,
            lookup,
            minimum=min(100, len(dataset)),
            output_dir=audit_dir,
        )
        if not pairings[split]["passed"]:
            raise RuntimeError(f"{split} pairing audit failed")
    probe = CNNTextToBrainModel(GenerativeTextToAELatent(), autoencoder)
    raw_path = audit_raw_latent_path(probe)
    if not raw_path["passed"]:
        raise RuntimeError(f"Raw-latent path audit failed: {raw_path}")
    return {
        "autoencoder": ae_source,
        "autoencoder_identity": autoencoder_identity(
            autoencoder,
            checkpoint=ae_path,
            domain=spec["domain"],
            branch=spec["branch"],
        ),
        "text_cache": {**cache_source, **text_cache_identity(lookup)},
        "text_preprocessing_audit": text_audit,
        "pairing_audits": pairings,
        "splits": {
            split: split_fingerprint(getattr(provider, split))
            for split in ("train", "val", "test")
        },
        "branch": dict(spec),
        "latent_convention": "raw_384d_stage1_ae_latent",
        "decoder_output": "raw_unclamped_for_training",
        "git_commit": RESOLVED_COMMIT,
    }

def atomic_torch_save(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{path.name}.", suffix=".tmp", dir=path.parent
    )
    os.close(descriptor)
    temporary = Path(temporary_name)
    try:
        torch.save(payload, temporary)
        os.replace(temporary, path)
    except BaseException:
        temporary.unlink(missing_ok=True)
        raise

def load_or_encode_train_latents(
    branch_dir, provenance, autoencoder, provider, lookup
):
    path = branch_dir / "training_target_latents.pt"
    split_hash = provenance["splits"]["train"]["ordered_rows_sha256"]
    encoder_hash = provenance["autoencoder"]["encoder_state_sha256"]
    if path.exists():
        payload = torch.load(path, map_location="cpu", weights_only=True)
        if payload.get("split_sha256") != split_hash:
            raise ValueError("Cached latent ordered-split checksum mismatch")
        if payload.get("encoder_state_sha256") != encoder_hash:
            raise ValueError("Cached latent frozen-encoder checksum mismatch")
        latents = payload["latents"]
    else:
        latents = encode_stage1_latents(
            autoencoder,
            provider.train,
            lookup,
            device=DEVICE,
            batch_size=EVAL_BATCH_SIZE,
            num_workers=NUM_WORKERS,
        )
        atomic_torch_save(
            path,
            {
                "latents": latents,
                "split_sha256": split_hash,
                "encoder_state_sha256": encoder_hash,
            },
        )
    if len(latents) != len(provider.train):
        raise RuntimeError("Ordered training latent count mismatch")
    return latents

## 6. Run definitions and causal phase separation

Fast primary runs cross the complete requested architecture/LR/weight
decay/dropout grid under constant LR. Scheduler policies are compared
only after validation selects optimizer settings. The optional
standardized latent MSE phase has a distinct `loss_mode` and never
enters architecture-only conclusions.

In [ ]:
SWEEP_CONFIG = {
    "fast_sweep": FAST_SWEEP,
    "full_sweep": FULL_SWEEP,
    "branches": BRANCHES_TO_RUN,
    "architectures": ALL_ARCHITECTURES,
    "fast_architectures": FAST_ARCHITECTURES,
    "fast_learning_rates": FAST_LEARNING_RATES,
    "fast_weight_decays": FAST_WEIGHT_DECAYS,
    "fast_dropouts": FAST_DROPOUTS,
    "schedulers": SCHEDULERS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "optimizer": "AdamW",
    "lion_compared": False,
    "primary_loss": "raw_latent_mse_plus_raw_decoded_volume_mse",
    "secondary_loss": (
        "training-fitted standardized_latent_mse_plus_raw_decoded_volume_mse"
        if RUN_SECONDARY_STANDARDIZED_SWEEP
        else None
    ),
    "test_used_for_selection": False,
    "pareto_objectives": PARETO_OBJECTIVES,
    "rank_columns": RANK_COLUMNS,
    "seed": SEED,
    "projector_seed": PROJECTOR_SEED,
    "pinned_commit": RESOLVED_COMMIT,
}
atomic_write_json(EXPERIMENT_ROOT / "sweep_config.json", SWEEP_CONFIG)

@dataclass(frozen=True)
class SweepRun:
    phase: str
    branch: str
    architecture: str
    learning_rate: float
    weight_decay: float
    dropout: float
    scheduler: str = "constant"
    loss_mode: str = "raw"
    epochs: int = EPOCHS

    def effective(self):
        return {
            **asdict(self),
            "optimizer": "AdamW",
            "input_dim": 768,
            "output_dim": 384,
            "raw_decoder_output": True,
            "final_output_transform": None,
            "latent_weight": 1.0,
            "reconstruction_weight": 1.0,
            "gradient_clip": GRADIENT_CLIP,
            "batch_size": BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "amp_dtype": str(MIXED_PRECISION_DTYPE),
            "seed": SEED,
            "projector_seed": PROJECTOR_SEED,
            "residual_2048_blocks": RESIDUAL_2048_BLOCKS,
            "test_used_for_selection": False,
        }

    @property
    def run_id(self):
        digest = sha256_value(self.effective())[:12]
        return (
            f"{self.phase}__{self.architecture}__lr{self.learning_rate:g}"
            f"__wd{self.weight_decay:g}__do{self.dropout:g}"
            f"__{self.scheduler}__{self.loss_mode}__{digest}"
        )

def fast_primary_runs():
    return [
        SweepRun(
            phase="fast_primary",
            branch="mixed_to_pubmed",
            architecture=architecture,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            dropout=dropout,
        )
        for architecture, learning_rate, weight_decay, dropout in itertools.product(
            FAST_ARCHITECTURES,
            FAST_LEARNING_RATES,
            FAST_WEIGHT_DECAYS,
            FAST_DROPOUTS,
        )
    ]

def lexicographic_rank(frame):
    available = [column for column in RANK_COLUMNS if column in frame]
    return frame.sort_values(
        available,
        ascending=[False] * len(available),
        kind="stable",
    )

def selected_fast_settings(frame, *, per_architecture):
    primary = frame[
        (frame["phase"] == "fast_primary") & (frame["loss_mode"] == "raw")
    ]
    selected = []
    for architecture, group in primary.groupby("architecture", sort=False):
        for row in lexicographic_rank(group).head(per_architecture).to_dict("records"):
            selected.append(
                {
                    "architecture": architecture,
                    "learning_rate": float(row["learning_rate"]),
                    "weight_decay": float(row["weight_decay"]),
                    "dropout": float(row["dropout"]),
                    "scheduler": "constant",
                }
            )
    return selected

def scheduler_runs(settings):
    return [
        SweepRun(
            phase="fast_scheduler",
            branch="mixed_to_pubmed",
            architecture=setting["architecture"],
            learning_rate=setting["learning_rate"],
            weight_decay=setting["weight_decay"],
            dropout=setting["dropout"],
            scheduler=scheduler,
        )
        for setting in settings
        for scheduler in SCHEDULERS
    ]

def secondary_runs(settings):
    return [
        SweepRun(
            phase="secondary_standardized",
            branch="mixed_to_pubmed",
            architecture=setting["architecture"],
            learning_rate=setting["learning_rate"],
            weight_decay=setting["weight_decay"],
            dropout=setting["dropout"],
            scheduler=setting["scheduler"],
            loss_mode="standardized",
        )
        for setting in settings
    ]

def load_full_settings():
    if SELECTED_OPTIMIZER_SETTINGS is not None:
        return list(SELECTED_OPTIMIZER_SETTINGS)
    source = (
        Path(FAST_RESULTS_DIR_FOR_FULL)
        if FAST_RESULTS_DIR_FOR_FULL is not None
        else EXPERIMENT_ROOT
    ) / "fast_selected_optimizer_settings.json"
    if not source.exists():
        raise FileNotFoundError(
            "FULL_SWEEP requires FAST_RESULTS_DIR_FOR_FULL pointing to a "
            "completed fast sweep, or SELECTED_OPTIMIZER_SETTINGS."
        )
    return json.loads(source.read_text())

def full_runs(settings):
    unique = []
    for setting in settings:
        candidate = {
            key: setting[key]
            for key in ("learning_rate", "weight_decay", "dropout", "scheduler")
        }
        if candidate not in unique:
            unique.append(candidate)
    unique = unique[:FULL_SELECTED_SETTING_COUNT]
    return [
        SweepRun(
            phase="full_primary",
            branch=branch,
            architecture=architecture,
            **setting,
            loss_mode="raw",
            epochs=FULL_EPOCHS,
        )
        for branch, architecture, setting in itertools.product(
            BRANCHES_TO_RUN, ALL_ARCHITECTURES, unique
        )
    ]

## 7. Loss, evaluation, semantic AUC, and diagnostics

Raw projector output is never normalized, squashed, or clipped. Clamping
is confined to the repository spatial metrics and released semantic
encoder convention. Validation metrics include every requested latent,
spatial, foreground, and semantic diagnostic.

In [ ]:
def compute_primary_loss(
    predicted_raw, target_raw, prediction_volume, target_volume,
    *, loss_mode, latent_mean, latent_std,
):
    raw_latent_mse = F.mse_loss(predicted_raw.float(), target_raw.float())
    if loss_mode == "raw":
        optimized_latent_mse = raw_latent_mse
    elif loss_mode == "standardized":
        optimized_latent_mse = F.mse_loss(
            (predicted_raw.float() - latent_mean) / latent_std,
            (target_raw.float() - latent_mean) / latent_std,
        )
    else:
        raise ValueError(f"Unknown loss mode: {loss_mode}")
    reconstruction_mse = F.mse_loss(
        prediction_volume.float(), target_volume.float()
    )
    total = optimized_latent_mse + reconstruction_mse
    return total, {
        "loss": total,
        "raw_latent_mse": raw_latent_mse,
        "optimized_latent_mse": optimized_latent_mse,
        "reconstruction_mse": reconstruction_mse,
    }

def semantic_metrics(
    semantic_model, brain_embeddings, text_embeddings,
    raw_text_embeddings, ids,
):
    n_examples = sum(len(value) for value in brain_embeddings)
    if n_examples < 2:
        return {
            "semantic_normalized_auc": float("nan"),
            "semantic_n": n_examples,
        }
    neighbors = max(0, min(SEMANTIC_NEIGHBORS, n_examples - 2))
    metrics, _ = evaluate_semantic_neighbor_retrieval(
        torch.cat(brain_embeddings),
        torch.cat(text_embeddings),
        ids,
        neighbor_text_embeddings=torch.cat(raw_text_embeddings),
        n_neighbors=neighbors,
    )
    return {
        "semantic_normalized_auc": float(
            metrics["semantic_normalized_k_recall_curve_auc"]
        ),
        "semantic_n": int(metrics["n_queries"]),
    }

@torch.no_grad()
def evaluate_projector(
    projector, autoencoder, semantic_model, dataset, lookup,
    train_latents, raw_transform, *, split,
):
    projector.eval()
    autoencoder.eval()
    semantic_model.eval()
    target_latents, prediction_latents = [], []
    spatial_totals = {}
    semantic_brain, semantic_text, semantic_raw_text, semantic_ids = [], [], [], []
    n = 0
    loader = make_loader(
        dataset, lookup, batch_size=EVAL_BATCH_SIZE,
        shuffle=False, seed=SEED,
    )
    for batch in loader:
        target = batch["volume"].to(DEVICE, non_blocking=True)
        text = batch["text_embedding"].to(DEVICE, non_blocking=True)
        target_raw = autoencoder.encoder(target)
        predicted_raw = projector(text)
        prediction = autoencoder.decoder(predicted_raw)
        spatial = reconstruction_metrics(prediction, target)
        batch_n = len(target)
        for name, value in spatial.items():
            spatial_totals[name] = (
                spatial_totals.get(name, 0.0) + float(value) * batch_n
            )
        target_latents.append(target_raw.float().cpu())
        prediction_latents.append(predicted_raw.float().cpu())
        n += batch_n
        used = sum(len(value) for value in semantic_brain)
        take = min(batch_n, max(0, SEMANTIC_MAX_EXAMPLES - used))
        if take:
            semantic_prediction = torch.nan_to_num(
                prediction[:take].float(),
                nan=0.0,
                posinf=1.0,
                neginf=0.0,
            ).clamp(0, 1)
            semantic_brain.append(
                semantic_model.encode_brain(semantic_prediction).float().cpu()
            )
            semantic_text.append(
                semantic_model.encode_text(text[:take]).float().cpu()
            )
            semantic_raw_text.append(text[:take].float().cpu())
            semantic_ids.extend(
                str(value) for value in batch["map_id"][:take]
            )
    if n < 2:
        raise RuntimeError(f"{split} evaluation needs at least two examples")
    latent_summary, per_dimension = latent_ablation_metrics(
        torch.cat(target_latents),
        torch.cat(prediction_latents),
        transform=raw_transform,
        nearest_reference=train_latents,
        distance_device=DEVICE,
    )
    summary = {
        **latent_summary,
        **{
            name: value / n
            for name, value in spatial_totals.items()
        },
        **semantic_metrics(
            semantic_model,
            semantic_brain,
            semantic_text,
            semantic_raw_text,
            semantic_ids,
        ),
    }
    summary.update(
        {
            "latent_variance_ratio": summary[
                "predicted_target_latent_variance_ratio"
            ],
            "latent_norm_ratio": summary[
                "predicted_target_latent_norm_ratio"
            ],
            "mean_dimension_r_squared": summary[
                "mean_per_dimension_r_squared"
            ],
            "top_quartile_dimension_r_squared": summary[
                "highest_target_variance_quartile_mean_r_squared"
            ],
            "dimension_variance_correlation": summary[
                "target_prediction_per_dimension_variance_correlation"
            ],
            "spatial_correlation": summary["spatial_corr"],
            "top5_dice": summary["top5_dice"],
            "foreground_mse": summary["foreground_mse"],
        }
    )
    return summary, per_dimension

def parameter_norm(module):
    return math.sqrt(
        sum(float(parameter.detach().float().square().sum())
            for parameter in module.parameters())
    )

## 8. Resume-safe training and strict checkpoints

Every run writes its effective config, provenance, architecture
definition, histories, layer activation/gradient diagnostics, best and
last checkpoints, and a manifest. Resume rejects changes to architecture,
frozen AE/text/splits, effective config, or projector state.

In [ ]:
def checkpoint_binding(provenance):
    return {
        "autoencoder": provenance["autoencoder"],
        "text_cache": provenance["text_cache"],
        "splits": provenance["splits"],
        "branch": provenance["branch"],
        "latent_convention": provenance["latent_convention"],
        "git_commit": provenance["git_commit"],
    }

def save_checkpoint(
    path, projector, optimizer, scheduler, scaler, run,
    build_config, binding, epoch, metrics,
):
    effective = run.effective()
    metadata = projector_checkpoint_metadata(
        build_config,
        projector,
        binding=binding,
        effective_config=effective,
    )
    payload = {
        "format_version": 1,
        "projector_state_dict": projector.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": (
            scheduler.state_dict() if scheduler is not None else None
        ),
        "scaler_state_dict": scaler.state_dict(),
        "epoch": epoch,
        "metrics": dict(metrics),
        "metadata": metadata,
        "effective_config": effective,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }
    atomic_torch_save(path, payload)
    return {
        "path": str(path),
        "sha256": sha256_file(path),
        "epoch": epoch,
        "projector_state_sha256": metadata["projector_state_sha256"],
        "metrics": dict(metrics),
    }

def load_checkpoint(
    path, projector, optimizer, scheduler, scaler, run,
    build_config, binding,
):
    path = Path(path)
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if payload.get("effective_config") != run.effective():
        raise ValueError("Checkpoint effective config mismatch")
    projector.load_state_dict(payload["projector_state_dict"], strict=True)
    validate_projector_checkpoint_metadata(
        payload["metadata"],
        build_config,
        binding=binding,
        effective_config=run.effective(),
        module=projector,
    )
    if optimizer is not None:
        optimizer.load_state_dict(payload["optimizer_state_dict"])
    if scheduler is not None:
        if payload.get("scheduler_state_dict") is None:
            raise ValueError("Checkpoint is missing scheduler state")
        scheduler.load_state_dict(payload["scheduler_state_dict"])
    scaler.load_state_dict(payload.get("scaler_state_dict") or {})
    torch.set_rng_state(payload["torch_rng_state"])
    if torch.cuda.is_available() and payload.get("cuda_rng_state_all"):
        torch.cuda.set_rng_state_all(payload["cuda_rng_state_all"])
    return payload

def train_epoch(
    projector, autoencoder, dataset, lookup, train_latents,
    optimizer, scaler, run, epoch, latent_mean, latent_std,
):
    projector.train()
    autoencoder.eval()
    totals = {}
    gradient_layer_totals = {}
    n = 0
    projected_sum = projected_square_sum = projected_count = 0.0
    autocast_enabled = (
        DEVICE.type == "cuda" and MIXED_PRECISION_DTYPE != torch.float32
    )
    loader = make_loader(
        dataset, lookup, batch_size=BATCH_SIZE,
        shuffle=True, seed=SEED + epoch,
    )
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    epoch_started = time.perf_counter()
    with ActivationMonitor(projector) as activation_monitor:
        for batch in loader:
            target = batch["volume"].to(DEVICE, non_blocking=True)
            text = batch["text_embedding"].to(DEVICE, non_blocking=True)
            indices = torch.as_tensor(batch["dataset_index"], dtype=torch.long)
            target_raw = train_latents.index_select(0, indices).to(
                DEVICE, non_blocking=True
            )
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=MIXED_PRECISION_DTYPE,
                enabled=autocast_enabled,
            ):
                predicted_raw = projector(text)
                prediction = autoencoder.decoder(predicted_raw)
                loss, parts = compute_primary_loss(
                    predicted_raw,
                    target_raw,
                    prediction,
                    target,
                    loss_mode=run.loss_mode,
                    latent_mean=latent_mean,
                    latent_std=latent_std,
                )
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
            else:
                loss.backward()
            gradient_summary, gradient_rows = gradient_diagnostics(projector)
            before = clone_trainable_parameters(projector)
            if GRADIENT_CLIP is not None:
                torch.nn.utils.clip_grad_norm_(
                    projector.parameters(), GRADIENT_CLIP
                )
            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            update_norm = parameter_update_norm(projector, before)
            batch_n = len(target)
            batch_metrics = {
                **{
                    name: float(value.detach())
                    for name, value in parts.items()
                },
                **gradient_summary,
                "parameter_update_norm": update_norm,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
            }
            for name, value in batch_metrics.items():
                totals[name] = totals.get(name, 0.0) + value * batch_n
            for row in gradient_rows:
                values = gradient_layer_totals.setdefault(
                    row["layer"],
                    {
                        "gradient_norm": 0.0,
                        "zero_gradient_percent": 0.0,
                        "batches": 0,
                    },
                )
                values["gradient_norm"] += row["gradient_norm"]
                values["zero_gradient_percent"] += row[
                    "zero_gradient_percent"
                ]
                values["batches"] += 1
            values = predicted_raw.detach().float()
            projected_sum += float(values.sum())
            projected_square_sum += float(values.square().sum())
            projected_count += values.numel()
            n += batch_n
    if not n:
        raise RuntimeError("Training loader produced no batches")
    mean = projected_sum / projected_count
    std = math.sqrt(
        max(projected_square_sum / projected_count - mean * mean, 0.0)
    )
    summary = {name: value / n for name, value in totals.items()}
    summary.update(
        {
            "projected_latent_mean": mean,
            "projected_latent_std": std,
            "parameter_norm": parameter_norm(projector),
            "peak_gpu_memory_bytes": (
                int(torch.cuda.max_memory_allocated())
                if torch.cuda.is_available()
                else 0
            ),
            "epoch_time_seconds": time.perf_counter() - epoch_started,
        }
    )
    activation_rows = activation_monitor.summary()
    required_activation_fields = {
        "activation_mean", "activation_std", "activation_zero_percent"
    }
    if any(
        not required_activation_fields <= row.keys()
        for row in activation_rows
    ):
        raise RuntimeError("Activation monitor schema is incomplete")
    layer_rows = [
        {
            "epoch": epoch,
            "diagnostic_type": "activation",
            **row,
        }
        for row in activation_rows
    ]
    layer_rows.extend(
        {
            "epoch": epoch,
            "diagnostic_type": "gradient",
            "layer": name,
            "gradient_norm": values["gradient_norm"] / values["batches"],
            "zero_gradient_percent": (
                values["zero_gradient_percent"] / values["batches"]
            ),
            "batches": values["batches"],
        }
        for name, values in gradient_layer_totals.items()
    )
    summary["max_layer_activation_zero_percent"] = max(
        (
            row["activation_zero_percent"]
            for row in activation_rows
            if row["layer_type"] in {"ReLU", "GELU"}
        ),
        default=0.0,
    )
    summary["max_layer_activation_abs"] = max(
        (row["activation_max_abs"] for row in activation_rows),
        default=0.0,
    )
    return summary, layer_rows, n

def train_one_run(
    run, autoencoder, semantic_model, provider, lookup,
    train_latents, provenance, eligible_architectures,
):
    if not eligible_architectures.get(run.architecture, False):
        raise RuntimeError(
            f"{run.architecture} failed its tiny-set capacity gate"
        )
    run_dir = EXPERIMENT_ROOT / run.branch / "runs" / run.run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    effective = run.effective()
    binding = checkpoint_binding(provenance)
    summary_path = run_dir / "run_summary.json"
    if summary_path.exists():
        completed = json.loads(summary_path.read_text())
        if completed.get("state") == "completed":
            recorded_effective = json.loads(
                (run_dir / "effective_config.json").read_text()
            )
            recorded_provenance = json.loads(
                (run_dir / "provenance.json").read_text()
            )
            if recorded_effective != effective:
                raise ValueError("Completed run effective config mismatch")
            if sha256_value(
                checkpoint_binding(recorded_provenance)
            ) != sha256_value(binding):
                raise ValueError("Completed run provenance binding mismatch")
            return completed["leaderboard_row"]

    build_config = ProjectorBuildConfig(
        name=run.architecture,
        dropout=run.dropout,
        residual_2048_blocks=RESIDUAL_2048_BLOCKS,
    )
    atomic_write_json(run_dir / "effective_config.json", effective)
    atomic_write_json(run_dir / "provenance.json", provenance)
    atomic_write_json(
        run_dir / "architecture_definition.json",
        projector_definition(build_config),
    )

    seed_everything(PROJECTOR_SEED)
    projector = build_stage4_projector(
        run.architecture,
        dropout=run.dropout,
        residual_2048_blocks=RESIDUAL_2048_BLOCKS,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        projector.parameters(),
        lr=run.learning_rate,
        weight_decay=run.weight_decay,
    )
    scheduler = build_scheduler(
        optimizer, run.scheduler, epochs=run.epochs
    )
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            DEVICE.type == "cuda"
            and MIXED_PRECISION_DTYPE == torch.float16
        ),
    )
    raw_transform = LatentTransform.fit(train_latents, "raw").to("cpu")
    latent_mean = train_latents.float().mean(0).to(DEVICE)
    latent_std = train_latents.float().std(0, unbiased=False).clamp_min(1e-4).to(DEVICE)

    history_path = run_dir / "training_history.csv"
    validation_path = run_dir / "validation_history.csv"
    layer_path = run_dir / "layer_diagnostics.csv"
    history = (
        pd.read_csv(history_path).to_dict("records")
        if history_path.exists()
        else []
    )
    validations = (
        pd.read_csv(validation_path).to_dict("records")
        if validation_path.exists()
        else []
    )
    layer_rows = (
        pd.read_csv(layer_path).to_dict("records")
        if layer_path.exists()
        else []
    )
    manifest_path = run_dir / "checkpoint_manifest.json"
    manifest = (
        json.loads(manifest_path.read_text())
        if manifest_path.exists()
        else {"format_version": 1, "checkpoints": {}}
    )
    last = manifest["checkpoints"].get("last")
    if last:
        if sha256_file(last["path"]) != last["sha256"]:
            raise ValueError("Last checkpoint file checksum mismatch")
        resumed = load_checkpoint(
            last["path"], projector, optimizer, scheduler, scaler,
            run, build_config, binding,
        )
        start_epoch = int(resumed["epoch"]) + 1
    else:
        start_epoch = 1
    history = [
        row for row in history if int(row["epoch"]) < start_epoch
    ]
    validations = [
        row for row in validations if int(row["epoch"]) < start_epoch
    ]
    layer_rows = [
        row for row in layer_rows if int(row["epoch"]) < start_epoch
    ]
    best_top5 = max(
        (float(row["val_top5_dice"]) for row in validations),
        default=-float("inf"),
    )
    stale_epochs = 0
    run_started = time.perf_counter()

    for epoch in range(start_epoch, run.epochs + 1):
        train_summary, epoch_layer_rows, train_n = train_epoch(
            projector, autoencoder, provider.train, lookup,
            train_latents, optimizer, scaler, run, epoch,
            latent_mean, latent_std,
        )
        if epoch % VALIDATE_EVERY_EPOCHS:
            continue
        validation, per_dimension = evaluate_projector(
            projector, autoencoder, semantic_model, provider.val,
            lookup, train_latents, raw_transform, split="val",
        )
        step_scheduler(
            scheduler,
            run.scheduler,
            validation_loss=validation["raw_latent_mse"],
        )
        diagnostic_input = {
            **train_summary,
            "parameter_update_norm": train_summary[
                "parameter_update_norm"
            ],
            "training_raw_latent_mse": train_summary["raw_latent_mse"],
            "validation_raw_latent_mse": validation["raw_latent_mse"],
            "latent_norm_ratio": validation["latent_norm_ratio"],
        }
        flags = detect_training_pathologies(diagnostic_input)
        history_row = {
            "epoch": epoch,
            "n": train_n,
            **{
                f"train_{name}": value
                for name, value in train_summary.items()
            },
            **{f"flag_{name}": value for name, value in flags.items()},
            "total_wall_time_seconds": time.perf_counter() - run_started,
        }
        validation_row = {
            "epoch": epoch,
            "n": len(provider.val),
            **{f"val_{name}": value for name, value in validation.items()},
        }
        history.append(history_row)
        validations.append(validation_row)
        layer_rows.extend(epoch_layer_rows)
        atomic_write_csv(history_path, history)
        atomic_write_csv(validation_path, validations)
        atomic_write_csv(layer_path, layer_rows)
        atomic_write_csv(
            run_dir / "per_dimension_latent_diagnostics.csv",
            [{"epoch": epoch, **row} for row in per_dimension],
        )
        checkpoint_metrics = {**history_row, **validation_row}
        last_record = save_checkpoint(
            run_dir / "checkpoints" / "last.pt",
            projector, optimizer, scheduler, scaler, run,
            build_config, binding, epoch, checkpoint_metrics,
        )
        manifest["checkpoints"]["last"] = last_record
        top5 = float(validation["top5_dice"])
        if top5 > best_top5 + EARLY_STOPPING_MIN_DELTA:
            best_top5 = top5
            stale_epochs = 0
            manifest["checkpoints"]["best_validation_top5_dice"] = save_checkpoint(
                run_dir / "checkpoints" / "best_validation_top5_dice.pt",
                projector, optimizer, scheduler, scaler, run,
                build_config, binding, epoch, checkpoint_metrics,
            )
        else:
            stale_epochs += 1
        atomic_write_json(manifest_path, manifest)
        if (
            EARLY_STOPPING_PATIENCE is not None
            and stale_epochs >= EARLY_STOPPING_PATIENCE
        ):
            break

    best = manifest["checkpoints"].get("best_validation_top5_dice")
    if best is None:
        raise RuntimeError("No validation top-5 checkpoint was produced")
    payload = load_checkpoint(
        best["path"], projector, None, None, scaler,
        run, build_config, binding,
    )
    selected_validation, selected_per_dimension = evaluate_projector(
        projector, autoencoder, semantic_model, provider.val,
        lookup, train_latents, raw_transform, split="val",
    )
    atomic_write_csv(
        run_dir / "selected_validation_per_dimension.csv",
        selected_per_dimension,
    )
    selected_epoch = int(payload["epoch"])
    selected_train = next(
        row for row in history if int(row["epoch"]) == selected_epoch
    )
    total_wall = time.perf_counter() - run_started
    row = {
        "run_id": run.run_id,
        "phase": run.phase,
        "branch": run.branch,
        "domain": BRANCH_SPECS[run.branch]["domain"],
        "architecture": run.architecture,
        "learning_rate": run.learning_rate,
        "weight_decay": run.weight_decay,
        "dropout": run.dropout,
        "scheduler": run.scheduler,
        "loss_mode": run.loss_mode,
        "selected_epoch": selected_epoch,
        "parameter_count": count_parameters(projector),
        "train_raw_latent_mse": selected_train[
            "train_raw_latent_mse"
        ],
        "train_reconstruction_mse": selected_train[
            "train_reconstruction_mse"
        ],
        "total_gradient_norm": selected_train[
            "train_total_gradient_norm"
        ],
        "parameter_update_norm": selected_train[
            "train_parameter_update_norm"
        ],
        "learning_rate_at_selection": selected_train[
            "train_learning_rate"
        ],
        "projected_latent_mean": selected_train[
            "train_projected_latent_mean"
        ],
        "projected_latent_std": selected_train[
            "train_projected_latent_std"
        ],
        "peak_gpu_memory_bytes": max(
            float(item["train_peak_gpu_memory_bytes"])
            for item in history
        ),
        "mean_epoch_time_seconds": float(
            np.mean([item["train_epoch_time_seconds"] for item in history])
        ),
        "total_wall_time_seconds": total_wall,
        **{
            f"val_{name}": value
            for name, value in selected_validation.items()
        },
        "checkpoint_path": best["path"],
        "run_dir": str(run_dir),
        "test_evaluated": False,
    }
    atomic_write_json(
        summary_path,
        {
            "state": "completed",
            "leaderboard_row": row,
            "test_used_for_selection": False,
        },
    )
    del projector, optimizer, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

## 9. Identical 32-example capacity gates

Every architecture must reach 95% of the branch-specific frozen-AE top-5
ceiling before it may enter a dataset sweep. The table also records the
first step reaching 99%, final latent MSE/explained variance/norm ratio,
wall time, and peak memory. Dropout is disabled for this capacity test so
the architecture, rather than regularization noise, is gated.

In [ ]:
def tiny_capacity_test(
    architecture, autoencoder, dataset, lookup, train_latents
):
    batch = next(
        iter(
            make_loader(
                dataset, lookup, batch_size=TINY_OVERFIT_N,
                shuffle=False, seed=SEED,
            )
        )
    )
    text = batch["text_embedding"].to(DEVICE)
    target = batch["volume"].to(DEVICE)
    indices = torch.as_tensor(batch["dataset_index"], dtype=torch.long)
    target_raw = train_latents.index_select(0, indices).to(DEVICE)
    with torch.no_grad():
        ceiling_volume = autoencoder.decoder(target_raw)
        ceiling_top5 = reconstruction_metrics(
            ceiling_volume, target
        )["top5_dice"]
    seed_everything(PROJECTOR_SEED)
    projector = build_stage4_projector(
        architecture,
        dropout=0.0,
        residual_2048_blocks=RESIDUAL_2048_BLOCKS,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        projector.parameters(),
        lr=TINY_LEARNING_RATE,
        weight_decay=0.0,
    )
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    step_95 = step_99 = None
    history = []
    final_prediction = None
    for step in range(1, TINY_OVERFIT_STEPS + 1):
        projector.train()
        optimizer.zero_grad(set_to_none=True)
        predicted_raw = projector(text)
        prediction = autoencoder.decoder(predicted_raw)
        loss = (
            F.mse_loss(predicted_raw.float(), target_raw.float())
            + F.mse_loss(prediction.float(), target.float())
        )
        loss.backward()
        optimizer.step()
        if step % TINY_EVAL_EVERY == 0 or step == TINY_OVERFIT_STEPS:
            projector.eval()
            with torch.no_grad():
                final_prediction = projector(text)
                decoded = autoencoder.decoder(final_prediction)
                top5 = reconstruction_metrics(decoded, target)["top5_dice"]
                latent_mse = float(
                    F.mse_loss(final_prediction.float(), target_raw.float())
                )
            fraction = (
                top5 / ceiling_top5
                if ceiling_top5 > 0
                else float(top5 >= ceiling_top5)
            )
            history.append(
                {
                    "architecture": architecture,
                    "step": step,
                    "top5_dice": top5,
                    "ae_ceiling_top5_dice": ceiling_top5,
                    "ceiling_fraction": fraction,
                    "latent_mse": latent_mse,
                }
            )
            if step_95 is None and fraction >= 0.95:
                step_95 = step
            if step_99 is None and fraction >= 0.99:
                step_99 = step
    latent_summary, _ = latent_ablation_metrics(
        target_raw.float().cpu(),
        final_prediction.float().cpu(),
        transform=LatentTransform.fit(target_raw.float().cpu(), "raw"),
        nearest_reference=target_raw.float().cpu(),
    )
    return {
        "architecture": architecture,
        "steps_to_95_percent_ae_top5_ceiling": step_95,
        "steps_to_99_percent_ae_top5_ceiling": step_99,
        "final_latent_mse": latent_summary["raw_latent_mse"],
        "final_explained_variance": latent_summary[
            "global_explained_variance"
        ],
        "final_predicted_target_norm_ratio": latent_summary[
            "predicted_target_latent_norm_ratio"
        ],
        "ae_top5_ceiling": ceiling_top5,
        "final_top5_dice": history[-1]["top5_dice"],
        "wall_time_seconds": time.perf_counter() - started,
        "peak_gpu_memory_bytes": (
            int(torch.cuda.max_memory_allocated())
            if torch.cuda.is_available()
            else 0
        ),
        "passed": step_95 is not None,
        "history": history,
    }

def run_capacity_gates(
    branch_dir, provenance, autoencoder, provider, lookup, train_latents
):
    output_path = branch_dir / "tiny_capacity_results.json"
    capacity_config = {
        "n": TINY_OVERFIT_N,
        "steps": TINY_OVERFIT_STEPS,
        "eval_every": TINY_EVAL_EVERY,
        "learning_rate": TINY_LEARNING_RATE,
        "weight_decay": 0.0,
        "dropout": 0.0,
        "seed": PROJECTOR_SEED,
        "pass_fraction": TINY_PASS_FRACTION,
        "architectures": ALL_ARCHITECTURES,
    }
    capacity_binding_sha256 = sha256_value(
        {
            "checkpoint_binding": checkpoint_binding(provenance),
            "capacity_config": capacity_config,
        }
    )
    if output_path.exists():
        saved = json.loads(output_path.read_text())
        if saved.get("binding_sha256") != capacity_binding_sha256:
            raise ValueError("Tiny-capacity result provenance/config mismatch")
        results = saved["results"]
    else:
        model_probe = CNNTextToBrainModel(
            GenerativeTextToAELatent(), autoencoder
        ).to(DEVICE)
        batch = next(
            iter(
                make_loader(
                    provider.train, lookup,
                    batch_size=TINY_OVERFIT_N,
                    shuffle=False, seed=SEED,
                )
            )
        )
        ceiling = ae_ceiling_bypass(
            model_probe, batch["volume"].to(DEVICE)
        )
        determinism = frozen_ae_determinism(
            model_probe, batch["volume"].to(DEVICE), repeats=3
        )
        if not ceiling["passed"] or not determinism["passed"]:
            raise RuntimeError(
                f"Frozen AE controls failed: {ceiling}, {determinism}"
            )
        results = [
            tiny_capacity_test(
                architecture, autoencoder, provider.train,
                lookup, train_latents,
            )
            for architecture in ALL_ARCHITECTURES
        ]
        atomic_write_json(
            output_path,
            {
                "binding_sha256": capacity_binding_sha256,
                "capacity_config": capacity_config,
                "results": results,
            },
        )
        atomic_write_csv(
            branch_dir / "tiny_capacity_history.csv",
            [
                row
                for result in results
                for row in result["history"]
            ],
        )
    rows = [
        {key: value for key, value in result.items() if key != "history"}
        for result in results
    ]
    atomic_write_csv(branch_dir / "tiny_capacity_table.csv", rows)
    return {row["architecture"]: bool(row["passed"]) for row in rows}

## 10. Execute the selected mode

This cell is resumable at branch and run level. Fast mode writes
validation-selected optimizer settings for the later full sweep. Full
mode applies those settings to every architecture and selected branch.
The test split is not touched here.

In [ ]:
lookup = AtlasFreeTextEmbeddingLookup.published()
leaderboard_rows = []
all_provenance = {}
branch_capacity = {}

initial_runs = fast_primary_runs() if FAST_SWEEP else full_runs(load_full_settings())
runs_by_branch = {
    branch: [run for run in initial_runs if run.branch == branch]
    for branch in BRANCHES_TO_RUN
}

for branch_name in BRANCHES_TO_RUN:
    print(f"===== {branch_name} =====")
    branch_dir = EXPERIMENT_ROOT / branch_name
    branch_dir.mkdir(parents=True, exist_ok=True)
    audit_dir = branch_dir / "provenance_audits"
    audit_dir.mkdir(exist_ok=True)
    spec, ae_path, autoencoder, provider, semantic_model = load_branch_resources(
        branch_name
    )
    provenance = build_branch_provenance(
        spec, ae_path, autoencoder, provider, lookup, audit_dir
    )
    all_provenance[branch_name] = provenance
    atomic_write_json(branch_dir / "provenance.json", provenance)
    train_latents = load_or_encode_train_latents(
        branch_dir, provenance, autoencoder, provider, lookup
    )
    eligible = run_capacity_gates(
        branch_dir, provenance, autoencoder, provider, lookup, train_latents
    )
    branch_capacity[branch_name] = eligible

    for run in runs_by_branch[branch_name]:
        print(run.run_id)
        leaderboard_rows.append(
            train_one_run(
                run, autoencoder, semantic_model, provider, lookup,
                train_latents, provenance, eligible,
            )
        )
        atomic_write_csv(
            EXPERIMENT_ROOT / "validation_leaderboard.csv",
            leaderboard_rows,
        )

    if FAST_SWEEP:
        primary_frame = pd.DataFrame(leaderboard_rows)
        chosen = selected_fast_settings(
            primary_frame,
            per_architecture=FAST_SELECTED_PER_ARCHITECTURE,
        )
        if RUN_SCHEDULER_COMPARISON:
            for run in scheduler_runs(chosen):
                print(run.run_id)
                leaderboard_rows.append(
                    train_one_run(
                        run, autoencoder, semantic_model, provider,
                        lookup, train_latents, provenance, eligible,
                    )
                )
                atomic_write_csv(
                    EXPERIMENT_ROOT / "validation_leaderboard.csv",
                    leaderboard_rows,
                )
        current = pd.DataFrame(leaderboard_rows)
        raw_candidates = current[
            current["loss_mode"].eq("raw")
            & current["architecture"].isin(FAST_ARCHITECTURES)
        ]
        final_fast_settings = []
        for architecture, group in raw_candidates.groupby("architecture"):
            best = lexicographic_rank(group).iloc[0]
            final_fast_settings.append(
                {
                    "architecture": architecture,
                    "learning_rate": float(best.learning_rate),
                    "weight_decay": float(best.weight_decay),
                    "dropout": float(best.dropout),
                    "scheduler": str(best.scheduler),
                }
            )
        atomic_write_json(
            EXPERIMENT_ROOT / "fast_selected_optimizer_settings.json",
            final_fast_settings,
        )
        if RUN_SECONDARY_STANDARDIZED_SWEEP:
            for run in secondary_runs(final_fast_settings):
                leaderboard_rows.append(
                    train_one_run(
                        run, autoencoder, semantic_model, provider,
                        lookup, train_latents, provenance, eligible,
                    )
                )
                atomic_write_csv(
                    EXPERIMENT_ROOT / "validation_leaderboard.csv",
                    leaderboard_rows,
                )

    del autoencoder, semantic_model, provider, train_latents
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

atomic_write_json(EXPERIMENT_ROOT / "provenance.json", all_provenance)
atomic_write_json(
    EXPERIMENT_ROOT / "capacity_gate_manifest.json", branch_capacity
)
print("Validation-only sweep complete:", EXPERIMENT_ROOT)

## 11. Separate-column ranking, Pareto front, and finalist-only test

No opaque composite score is computed. The leaderboard retains top-5
Dice, spatial correlation, semantic normalized recall AUC, latent
variance ratio, and global explained variance as separate columns.
Pareto objectives are configurable above. Fast mode stops after
validation analysis; only full mode evaluates the test split, and then
only for validation-selected finalists.

In [ ]:
leaderboard = pd.read_csv(EXPERIMENT_ROOT / "validation_leaderboard.csv")
raw_primary = leaderboard[
    leaderboard["loss_mode"].eq("raw")
    & leaderboard["phase"].isin(["fast_primary", "fast_scheduler", "full_primary"])
].copy()
raw_primary["is_pareto"] = pareto_front(
    raw_primary.to_dict("records"),
    PARETO_OBJECTIVES,
    epsilon=PARETO_EPSILON,
)
ranked = lexicographic_rank(raw_primary)
atomic_write_csv(
    EXPERIMENT_ROOT / "validation_leaderboard.csv",
    leaderboard.to_dict("records"),
)
atomic_write_csv(
    EXPERIMENT_ROOT / "pareto_front.csv",
    raw_primary[raw_primary["is_pareto"]].to_dict("records"),
)
display(
    ranked[
        [
            "branch", "architecture", "learning_rate", "weight_decay",
            "dropout", "scheduler", *RANK_COLUMNS, "is_pareto",
        ]
    ]
)

test_rows = []
root_checkpoint_manifest = {"format_version": 1, "finalists": []}
if FULL_SWEEP:
    finalists = []
    for branch, group in raw_primary.groupby("branch", sort=False):
        pareto_group = group[group["is_pareto"]]
        pool = pareto_group if len(pareto_group) else group
        finalists.extend(
            lexicographic_rank(pool)
            .head(FINALISTS_PER_BRANCH)
            .to_dict("records")
        )
    for finalist in finalists:
        branch = finalist["branch"]
        spec, ae_path, autoencoder, provider, semantic_model = load_branch_resources(
            branch
        )
        provenance = all_provenance[branch]
        branch_dir = EXPERIMENT_ROOT / branch
        train_latents = load_or_encode_train_latents(
            branch_dir, provenance, autoencoder, provider, lookup
        )
        raw_transform = LatentTransform.fit(train_latents, "raw")
        run = SweepRun(
            phase=finalist["phase"],
            branch=branch,
            architecture=finalist["architecture"],
            learning_rate=float(finalist["learning_rate"]),
            weight_decay=float(finalist["weight_decay"]),
            dropout=float(finalist["dropout"]),
            scheduler=finalist["scheduler"],
            loss_mode=finalist["loss_mode"],
            epochs=FULL_EPOCHS,
        )
        build_config = ProjectorBuildConfig(
            name=run.architecture,
            dropout=run.dropout,
            residual_2048_blocks=RESIDUAL_2048_BLOCKS,
        )
        projector = build_stage4_projector(
            run.architecture,
            dropout=run.dropout,
            residual_2048_blocks=RESIDUAL_2048_BLOCKS,
        ).to(DEVICE)
        scaler = torch.amp.GradScaler("cuda", enabled=False)
        checkpoint = Path(finalist["checkpoint_path"])
        run_manifest = json.loads(
            (Path(finalist["run_dir"]) / "checkpoint_manifest.json").read_text()
        )
        recorded_best = run_manifest["checkpoints"][
            "best_validation_top5_dice"
        ]
        if Path(recorded_best["path"]) != checkpoint:
            raise ValueError("Finalist checkpoint path does not match run manifest")
        if sha256_file(checkpoint) != recorded_best["sha256"]:
            raise ValueError("Finalist checkpoint file checksum mismatch")
        payload = load_checkpoint(
            checkpoint,
            projector,
            None,
            None,
            scaler,
            run,
            build_config,
            checkpoint_binding(provenance),
        )
        test_summary, test_per_dimension = evaluate_projector(
            projector, autoencoder, semantic_model, provider.test,
            lookup, train_latents, raw_transform, split="test",
        )
        row = {
            "run_id": finalist["run_id"],
            "branch": branch,
            "architecture": run.architecture,
            "validation_selected_epoch": int(payload["epoch"]),
            **{f"test_{name}": value for name, value in test_summary.items()},
            "checkpoint_path": str(checkpoint),
        }
        test_rows.append(row)
        atomic_write_csv(
            Path(finalist["run_dir"]) / "test_results.csv", [row]
        )
        atomic_write_csv(
            Path(finalist["run_dir"]) / "test_per_dimension.csv",
            test_per_dimension,
        )
        root_checkpoint_manifest["finalists"].append(
            {
                "run_id": finalist["run_id"],
                "branch": branch,
                "checkpoint_path": str(checkpoint),
                "checkpoint_sha256": sha256_file(checkpoint),
                "selected_by": "validation Pareto then lexicographic columns",
                "test_used_for_selection": False,
            }
        )
        del projector, autoencoder, semantic_model, provider, train_latents
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    atomic_write_csv(EXPERIMENT_ROOT / "test_results_finalists_only.csv", test_rows)
else:
    atomic_write_csv(
        EXPERIMENT_ROOT / "test_results_finalists_only.csv",
        [{"status": "not_evaluated_in_fast_sweep"}],
    )
atomic_write_json(
    EXPERIMENT_ROOT / "checkpoint_manifest.json",
    root_checkpoint_manifest,
)

## 12. Collapse/architecture plots and final report

The report answers the six requested scientific questions from
validation comparisons. Secondary standardized-loss rows are displayed
separately and excluded from architecture-only claims.

In [ ]:
plots_dir = EXPERIMENT_ROOT / "plots"
plots_dir.mkdir(exist_ok=True)
raw = raw_primary.copy()
raw["architecture_setting"] = (
    raw["architecture"] + "\n" + raw["scheduler"].astype(str)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
best_by_arch = (
    raw.sort_values("val_top5_dice", ascending=False)
    .groupby(["branch", "architecture"], as_index=False)
    .first()
)
for branch, group in best_by_arch.groupby("branch"):
    axes[0].plot(
        group["architecture"], group["val_latent_variance_ratio"],
        marker="o", label=branch,
    )
    axes[1].plot(
        group["architecture"], group["val_latent_norm_ratio"],
        marker="o", label=branch,
    )
axes[0].axhline(1.0, color="black", ls="--")
axes[1].axhline(1.0, color="black", ls="--")
axes[0].set_title("Latent variance ratio (prediction / target)")
axes[1].set_title("Latent norm ratio (prediction / target)")
for axis in axes:
    axis.tick_params(axis="x", rotation=45)
    axis.legend(fontsize=8)
fig.tight_layout()
fig.savefig(plots_dir / "latent_collapse_plots.png", dpi=170)
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = [
    ("val_top5_dice", "Validation top-5 Dice"),
    ("val_spatial_corr", "Validation spatial correlation"),
    ("val_global_explained_variance", "Global explained variance"),
]
for axis, (metric, title) in zip(axes, metrics, strict=True):
    pivot = best_by_arch.pivot(
        index="architecture", columns="branch", values=metric
    )
    pivot.plot(kind="bar", ax=axis)
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(plots_dir / "architecture_comparison_plots.png", dpi=170)
plt.close(fig)

atomic_write_csv(
    EXPERIMENT_ROOT / "time_memory_table.csv",
    leaderboard[
        [
            "run_id", "branch", "architecture", "parameter_count",
            "selected_epoch", "mean_epoch_time_seconds",
            "total_wall_time_seconds", "peak_gpu_memory_bytes",
        ]
    ].to_dict("records"),
)

def best_row(frame):
    return lexicographic_rank(frame).iloc[0] if len(frame) else None

retained = raw[raw["architecture"] == "retained_mlp"]
larger = raw[raw["architecture"] != "retained_mlp"]
retained_best = best_row(retained)
larger_best = best_row(larger)
if retained_best is not None and larger_best is not None:
    train_gain = (
        float(retained_best.train_raw_latent_mse)
        - float(larger_best.train_raw_latent_mse)
    )
    validation_dice_gain = (
        float(larger_best.val_top5_dice)
        - float(retained_best.val_top5_dice)
    )
    variance_gain = (
        float(larger_best.val_latent_variance_ratio)
        - float(retained_best.val_latent_variance_ratio)
    )
    under_capacity = train_gain > 0 and validation_dice_gain > 0
else:
    train_gain = validation_dice_gain = variance_gain = float("nan")
    under_capacity = False

best_collapse = raw.sort_values(
    ["val_latent_variance_ratio", "val_global_explained_variance"],
    ascending=False,
).iloc[0]
domain_deltas = {}
for domain, group in best_by_arch.groupby(
    best_by_arch["branch"].map(lambda value: BRANCH_SPECS[value]["domain"])
):
    base = group[group["architecture"] == "retained_mlp"]
    alternatives = group[group["architecture"] != "retained_mlp"]
    if len(base) and len(alternatives):
        domain_deltas[domain] = float(
            alternatives["val_top5_dice"].max()
            - base["val_top5_dice"].max()
        )
sparse_values = [
    domain_deltas[name]
    for name in ("pubmed", "nilearn")
    if name in domain_deltas
]
sparse_gain = float(np.mean(sparse_values)) if sparse_values else float("nan")
neurovault_gain = domain_deltas.get("neurovault", float("nan"))
complexity_justified = bool(
    math.isfinite(validation_dice_gain)
    and validation_dice_gain >= COMPLEXITY_DICE_GAIN_THRESHOLD
)

standardized = leaderboard[
    leaderboard["loss_mode"].eq("standardized")
]
secondary_note = (
    "No standardized-latent secondary sweep was run."
    if standardized.empty
    else (
        "Standardized-latent results are in the leaderboard but are "
        "excluded from every architecture-only conclusion above."
    )
)
scope_note = (
    "The fast sweep cannot answer cross-domain question 5; run FULL_SWEEP."
    if FAST_SWEEP
    else (
        f"Mean PubMed/Nilearn Dice gain over retained: {sparse_gain:+.6f}; "
        f"NeuroVault gain: {neurovault_gain:+.6f}."
    )
)
report = f'''# Stage 4 projector architecture and optimization report

Pinned commit: `{RESOLVED_COMMIT}`  
Mode: `{"FAST_SWEEP" if FAST_SWEEP else "FULL_SWEEP"}`  
Optimizer: `AdamW` (Lion was not an allowed dependency)  
Test used for selection: `False`

## 1. Is the retained 768→512→384 projector under-capacity?

**{under_capacity}** under the prespecified criterion that a larger model must
improve both selected-epoch training raw-latent MSE and validation top-5
Dice. Best larger-minus-retained training-MSE improvement is
`{train_gain:+.6g}` and validation Dice improvement is
`{validation_dice_gain:+.6g}`.

## 2. Does a larger projector increase full-dataset latent variance?

Best larger-minus-retained validation latent variance-ratio change:
`{variance_gain:+.6g}`. The best collapse-control run is
`{best_collapse.architecture}` with variance ratio
`{float(best_collapse.val_latent_variance_ratio):.6g}` and global explained
variance `{float(best_collapse.val_global_explained_variance):.6g}`.

## 3. Which learning rate and weight decay prevent mean-latent collapse?

The best variance/explained-variance row uses learning rate
`{float(best_collapse.learning_rate):g}`, weight decay
`{float(best_collapse.weight_decay):g}`, dropout
`{float(best_collapse.dropout):g}`, and scheduler
`{best_collapse.scheduler}`. Spatial and semantic columns remain separate
in `validation_leaderboard.csv`; this is not an opaque-score winner.

## 4. Does capacity improve training only, or validation too?

Training raw-latent MSE improvement: `{train_gain:+.6g}`. Validation
top-5 Dice improvement: `{validation_dice_gain:+.6g}`. Therefore the
evidence is `{"training and validation" if train_gain > 0 and validation_dice_gain > 0 else "training only or inconclusive"}`.

## 5. PubMed/Nilearn versus NeuroVault

{scope_note}

## 6. Is the gain worth the added complexity?

**{complexity_justified}** using the configured absolute validation Dice
threshold `{COMPLEXITY_DICE_GAIN_THRESHOLD:g}`. Consult
`parameter_count_table.csv` and `time_memory_table.csv` for the exact
parameter, wall-time, and measured peak-memory costs.

## Secondary standardized-latent sweep

{secondary_note}

## Selection safeguards

- Ranking columns are retained separately; no composite score is used.
- `pareto_front.csv` uses the configured five validation objectives.
- Fast mode never evaluates test. Full mode evaluates test only for
  validation-selected finalists listed in `checkpoint_manifest.json`.
- The production Stage 4 projector and loader are unchanged.
'''
report = "\n".join(
    line[8:] if line.startswith("        ") else line
    for line in report.strip().splitlines()
) + "\n"
(EXPERIMENT_ROOT / "final_report.md").write_text(report)
print(report)

atomic_write_json(
    ACTIVE_POINTER,
    {
        "path": str(EXPERIMENT_ROOT),
        "state": "completed",
        "updated_at": utc_stamp(),
    },
)

## Artifact checklist

The experiment root contains:

- `sweep_config.json`
- per-run `effective_config.json`, `provenance.json`, and
  `architecture_definition.json`
- per-run `training_history.csv`, `validation_history.csv`,
  `layer_diagnostics.csv`, per-dimension diagnostics, and strict
  checkpoints
- `architecture_definitions.json`
- `parameter_count_table.csv`
- `validation_leaderboard.csv`
- `pareto_front.csv`
- `test_results_finalists_only.csv`
- `checkpoint_manifest.json`
- `time_memory_table.csv`
- `plots/latent_collapse_plots.png`
- `plots/architecture_comparison_plots.png`
- `final_report.md`

Tiny-capacity tables and histories are stored under each branch. A
failed architecture is hard-blocked from dataset training.